# COVID-19 Tweet Sentiment Classification

Academic team project: **Shadman Al Muksit, MD Tarek Hossen, and Sariful Islam**.

This is a binary **positive/negative sentiment** exercise, not misinformation detection. Positive and extremely positive labels are merged; negative and extremely negative labels are merged; neutral labels are excluded.

The original benchmark training and test files are combined and then split again into a new stratified 70/15/15 training/validation/test partition. Results therefore refer to this new random split, not the original benchmark test set. The regularization sweep exposes test scores for several C values, so that test set is not an untouched final evaluation after model selection.

This publication copy removes saved outputs, execution history, personal metadata, raw-text previews and example sentences. It has been checked for Python syntax, but the complete training workflow has not been re-executed for publication. See `../README.md` for setup, data provenance, archived results and limitations. Download the two CSV files separately; do not commit them.


Cell 1 — Imports

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

try:
    from google.colab import files
except ImportError:
    files = None

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Provide the two CSV files

In Colab, upload both files when prompted. Locally, place them in the repository `data/` directory. Raw data are intentionally excluded from this repository.

In [ ]:
if files is not None:
    uploaded = files.upload()
    print("Upload complete. Both Corona_NLP_train.csv and Corona_NLP_test.csv are required.")
else:
    print("Local mode: place both CSV files in the repository data/ directory.")


In [ ]:
required_names = ("Corona_NLP_train.csv", "Corona_NLP_test.csv")
if files is not None and not all(Path(name).is_file() for name in required_names):
    uploaded = files.upload()
    print("Additional upload complete.")


**Cell 3 — Load both datasets**

In [ ]:
def locate_dataset(filename):
    # A Colab upload is in the working directory. Local Jupyter may start at
    # either the repository root or the notebooks/ directory.
    candidates = [Path("data") / filename, Path("../data") / filename, Path(filename)]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Download the data separately and place both CSV files in data/."
    )

train_df = pd.read_csv(locate_dataset("Corona_NLP_train.csv"), encoding="latin1")
test_df = pd.read_csv(locate_dataset("Corona_NLP_test.csv"), encoding="latin1")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
# Raw text and account/location previews are deliberately not displayed.


Cell 4 — Null value check

In [ ]:
print("Train null values:\n")
print(train_df.isnull().sum())

print("\nTest null values:\n")
print(test_df.isnull().sum())

print("\nTotal nulls in train:", train_df.isnull().sum().sum())
print("Total nulls in test :", test_df.isnull().sum().sum())

**Cell 5 — Label add + combine dataset**

In [ ]:
# Add source column just to track where rows came from
train_df["source_file"] = "train"
test_df["source_file"] = "test"

# Combine datasets
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

print("Combined shape:", df.shape)
print(df["Sentiment"].value_counts(dropna=False))

**Cell 6 — Create binary label**

In [ ]:
# Keep only binary sentiment classes
positive_classes = ["Positive", "Extremely Positive"]
negative_classes = ["Negative", "Extremely Negative"]

df = df[df["Sentiment"].isin(positive_classes + negative_classes)].copy()

# Add numeric label
df["label"] = df["Sentiment"].apply(lambda x: 1 if x in positive_classes else 0)

print("Shape after keeping binary classes:", df.shape)
print("\nBinary label counts:")
print(df["label"].value_counts())
print("\nSentiment counts:")
print(df["Sentiment"].value_counts())

**Cell 7 — Cleaning + combine text columns**

In [ ]:
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)   # remove urls
    text = re.sub(r"@\w+", " ", text)                      # remove mentions
    text = re.sub(r"#", " ", text)                         # remove hashtag sign only
    text = re.sub(r"\d+", " ", text)                       # remove digits
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

# Null handling
df["Location"] = df["Location"].fillna("")

# Combine columns into one text field
df["combined_text"] = (df["Location"] + " " + df["OriginalTweet"]).str.strip()

# Clean the combined text
df["clean_text"] = df["combined_text"].apply(clean_text)

# Remove empty rows after cleaning
df = df[df["clean_text"].str.len() > 0].copy()

# Drop duplicates
df = df.drop_duplicates(subset=["clean_text", "label"]).reset_index(drop=True)

print("Shape after cleaning:", df.shape)
# Raw text previews are omitted from the publication copy.


**Cell 8 — Quick Exploratory Data Analysis**

In [ ]:
df["char_len"] = df["OriginalTweet"].astype(str).apply(len)
df["word_count"] = df["OriginalTweet"].astype(str).apply(lambda x: len(x.split()))

print(df.groupby("label")[["char_len", "word_count"]].mean())

plt.figure(figsize=(6,4))
df["label"].value_counts().sort_index().plot(kind="bar")
plt.title("Class Distribution (0=Negative, 1=Positive)")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8,4))
plt.hist(df[df["label"] == 0]["char_len"], bins=40, alpha=0.7, label="Negative")
plt.hist(df[df["label"] == 1]["char_len"], bins=40, alpha=0.7, label="Positive")
plt.title("Tweet Length Distribution")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.legend()
plt.show()

**Cell 9 — Train / Validation / Test split**

**Evaluation scope:** These splits are created after combining the original benchmark training and test files. They are new random partitions and do not preserve the benchmark holdout or enforce author/time separation.

In [ ]:
X = df["clean_text"]
y = df["label"]

# First split: train 70%, temp 30%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y #randomly selects the split; keeps the original pos and neg balance
)

# Second split: val 15%, test 15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp #randomly selects the split; keeps the original pos and neg balance
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

**Cell 10 — TF-IDF**

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Val TF-IDF shape  :", X_val_tfidf.shape)
print("Test TF-IDF shape :", X_test_tfidf.shape)

**Cell 11 — Logistic Regression baseline**

In [ ]:
baseline_model = LogisticRegression(  #Logistic Regression is being used to learn the relationship between TF-IDF words and sentiment labels.
    penalty="l2",   #Regularization L2
    C=1.0,          #Controls strength of regularization
    max_iter=3000,
    random_state=42
)

baseline_model.fit(X_train_tfidf, y_train)

**Cell 12 — Evaluation helper**

In [ ]:
def evaluate_model(model, X_data, y_true, name="Dataset"): #Performance Test
    y_pred = model.predict(X_data)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{name} Results")
    print("-" * 40)
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1-score :", round(f1, 4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "dataset": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }

**Cell 13 — Train / Validation / Test evaluation**

In [ ]:
train_result = evaluate_model(baseline_model, X_train_tfidf, y_train, "Train")
val_result   = evaluate_model(baseline_model, X_val_tfidf, y_val, "Validation")
test_result  = evaluate_model(baseline_model, X_test_tfidf, y_test, "Test")

print("\nClassification Report on Test Set:\n")
print(classification_report(
    y_test,
    baseline_model.predict(X_test_tfidf),
    target_names=["Negative", "Positive"],
    zero_division=0
))

**Cell 14 — Overfitting / Underfitting analysis**

In [ ]:
configs = [
    {"max_features": 200,  "C": 0.01},
    {"max_features": 500,  "C": 0.1},
    {"max_features": 1000, "C": 1.0},
    {"max_features": 3000, "C": 1.0},
    {"max_features": 5000, "C": 10.0},
]

analysis_rows = []

for cfg in configs:
    vec = TfidfVectorizer(max_features=cfg["max_features"], ngram_range=(1,2), min_df=2)

    Xtr = vec.fit_transform(X_train)
    Xv  = vec.transform(X_val)

    model = LogisticRegression(
        penalty="l2",
        C=cfg["C"],
        max_iter=3000,
        random_state=42
    )
    model.fit(Xtr, y_train)

    train_pred = model.predict(Xtr)
    val_pred   = model.predict(Xv)

    train_f1 = f1_score(y_train, train_pred, zero_division=0)
    val_f1   = f1_score(y_val, val_pred, zero_division=0)
    gap = train_f1 - val_f1

    if train_f1 < 0.75 and val_f1 < 0.75:
        remark = "Underfitting"
    elif gap > 0.04:
        remark = "Overfitting tendency"
    else:
        remark = "Good balance"

    analysis_rows.append([
        cfg["max_features"], cfg["C"],
        round(train_f1, 4), round(val_f1, 4), round(gap, 4), remark
    ])

analysis_df = pd.DataFrame(
    analysis_rows,
    columns=["max_features", "C", "train_f1", "val_f1", "gap", "remark"]
)

print(analysis_df)

plt.figure(figsize=(8,4))
plt.plot(analysis_df["max_features"], analysis_df["train_f1"], marker="o", label="Train F1")
plt.plot(analysis_df["max_features"], analysis_df["val_f1"], marker="o", label="Validation F1")
plt.title("Overfitting / Underfitting Analysis")
plt.xlabel("max_features")
plt.ylabel("F1-score")
plt.legend()
plt.show()

**Cell 15 — L2 regularization comparison**

**Historical experiment caveat:** The following original exercise reports test F1 for every C value. Do not select a model from those test scores and then describe that test set as untouched. For a new experiment, select hyperparameters on training/validation data and evaluate once on a separate final holdout.

In [ ]:
C_values = [0.01, 0.1, 1, 10, 100]
l2_rows = []

for c in C_values:
    model = LogisticRegression(
        penalty="l2",
        C=c,
        max_iter=3000,
        random_state=42
    )
    model.fit(X_train_tfidf, y_train)

    train_pred = model.predict(X_train_tfidf)
    val_pred   = model.predict(X_val_tfidf)
    test_pred  = model.predict(X_test_tfidf)

    l2_rows.append([
        c,
        round(f1_score(y_train, train_pred, zero_division=0), 4),
        round(f1_score(y_val, val_pred, zero_division=0), 4),
        round(f1_score(y_test, test_pred, zero_division=0), 4)
    ])

l2_df = pd.DataFrame(l2_rows, columns=["C", "Train_F1", "Val_F1", "Test_F1"])
print(l2_df)

plt.figure(figsize=(8,4))
plt.plot(l2_df["C"], l2_df["Train_F1"], marker="o", label="Train F1")
plt.plot(l2_df["C"], l2_df["Val_F1"], marker="o", label="Val F1")
plt.plot(l2_df["C"], l2_df["Test_F1"], marker="o", label="Test F1")
plt.xscale("log")
plt.title("L2 Regularization Comparison")
plt.xlabel("C (inverse regularization strength)")
plt.ylabel("F1-score")
plt.legend()
plt.show()

**Cell 16 — Prepare smaller TF-IDF for manual forward/backward propagation**

In [ ]:
manual_tfidf = TfidfVectorizer(   #To convert test/val/train texts into values
    max_features=600,
    ngram_range=(1, 1),
    min_df=2
)

X_train_manual = manual_tfidf.fit_transform(X_train).toarray().astype(np.float32)
X_val_manual   = manual_tfidf.transform(X_val).toarray().astype(np.float32)
X_test_manual  = manual_tfidf.transform(X_test).toarray().astype(np.float32)

y_train_manual = y_train.values.astype(np.float32)
y_val_manual   = y_val.values.astype(np.float32)
y_test_manual  = y_test.values.astype(np.float32)

print(X_train_manual.shape, X_val_manual.shape, X_test_manual.shape)

**Cell 17 — Manual forward propagation / backward propagation functions**

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)        #ranges between -500 to 500 so that doesn't oveflow
    return 1 / (1 + np.exp(-z))

def forward_propagation(X, w, b):    #takes X(tf-idf) with w(importance) to handle complexity
    z = np.dot(X, w) + b
    a = sigmoid(z)
    return z, a                      #'z' is raw nd 'a' is probability

def compute_loss(y_true, y_prob, w, lambda_): #real lebels; predicted probability
    m = len(y_true)                           #number of training examples
    eps = 1e-9
    data_loss = -np.mean(
        y_true * np.log(y_prob + eps) + (1 - y_true) * np.log(1 - y_prob + eps) #binary cross-entropy loss
    )
    reg_loss = (lambda_ / (2 * m)) * np.sum(w ** 2)  #L2 regularization loss
    return data_loss + reg_loss

def backward_propagation(X, y, a, w, lambda_):
    m = X.shape[0]
    dz = a - y                                       #predicted probability; actual label
    dw = (1 / m) * np.dot(X.T, dz) + (lambda_ / m) * w  #This calculates gradient of loss with respect to weights
    db = (1 / m) * np.sum(dz)                           #This calculates gradient of loss with respect to bias
    return dw, db

**Cell 18 — Train manual logistic regression with gradient descent**

In [ ]:
n_features = X_train_manual.shape[1]
w = np.zeros(n_features, dtype=np.float32)
b = 0.0

learning_rate = 0.5
epochs = 300
lambda_ = 0.1

train_losses = []
val_losses = []

for epoch in range(epochs):
    # Forward propagation
    z_train, a_train = forward_propagation(X_train_manual, w, b)
    train_loss = compute_loss(y_train_manual, a_train, w, lambda_)

    # Backward propagation
    dw, db = backward_propagation(X_train_manual, y_train_manual, a_train, w, lambda_)

    # Update
    w = w - learning_rate * dw  #subtracting LR*gradient_loss from prev_weight
    b = b - learning_rate * db  #subtracting LR*gradient_loss from prev_weight

    # Validation loss
    _, a_val = forward_propagation(X_val_manual, w, b)
    val_loss = compute_loss(y_val_manual, a_val, w, lambda_)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

**Cell 19 — Plot manual training curve**

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Manual Logistic Regression Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

**Cell 20 — Evaluate manual model**

In [ ]:
def predict_manual(X, w, b, threshold=0.5):
    _, probs = forward_propagation(X, w, b)
    preds = (probs >= threshold).astype(int)
    return probs, preds

def evaluate_manual(X, y_true, w, b, name="Dataset"):
    probs, preds = predict_manual(X, w, b)

    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)

    print(f"\n{name} Manual Model Results")
    print("-" * 40)
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1-score :", round(f1, 4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))

evaluate_manual(X_train_manual, y_train_manual, w, b, "Train")
evaluate_manual(X_val_manual, y_val_manual, w, b, "Validation")
evaluate_manual(X_test_manual, y_test_manual, w, b, "Test")

**Cell 21 — Final prediction**

In [ ]:
def predict_new_tweet(text, vectorizer, model):
    text = str(text)
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1]

    return {
        "original_text": text,
        "cleaned_text": cleaned,
        "predicted_label": "Positive" if pred == 1 else "Negative",
        "positive_probability": round(float(prob), 4)
    }

# Call this helper with your own permitted input after training.


**helper function**

In [ ]:
def predict_tweet(text, model, vectorizer):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])

    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1]

    label = "Positive" if pred == 1 else "Negative"
    return label, prob

**custom input UI code**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

title = widgets.HTML("<h3>COVID Tweet Sentiment Analysis System</h3>")

tweet_box = widgets.Textarea(
    placeholder='Paste a tweet or write any sentence here...',
    description='Input:',
    layout=widgets.Layout(width='850px', height='220px')
)

predict_button = widgets.Button(
    description='Check Sentiment',
    button_style='primary'
)

output = widgets.Output()

def on_predict_clicked(b):
    with output:
        clear_output()

        text = tweet_box.value.strip()

        if len(text) == 0:
            display(HTML("<b style='color:red;'>Please enter tweet text.</b>"))
            return

        label, prob = predict_tweet(text, baseline_model, tfidf)

        if label == "Positive":
            color = "green"
            msg = "This tweet is predicted as POSITIVE."
        else:
            color = "red"
            msg = "This tweet is predicted as NEGATIVE."

        display(HTML(f"""
        <h4>Prediction Result</h4>
        <p><b>Label:</b> <span style="color:{color}; font-size:18px;">{label}</span></p>
        <p><b>Probability of being Positive:</b> {round(prob, 4)}</p>
        <p><b>Message:</b> {msg}</p>
        """))

predict_button.on_click(on_predict_clicked)

display(title, tweet_box, predict_button, output)